# Encontro 5 · Dentro da avaliação: como um time ganha confiança para dar deploy

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/se-ai-agents-engineering/ai-agents-engineering-v2/blob/main/encontro-05/01-dentro-da-avaliacao.ipynb)

**Pergunta do dia:** não existe `assert` em linguagem natural. Como um time ganha confiança para dar deploy?

No Encontro 4, vocês decidiram a arquitetura com uma rodada do placar. Ficaram duas perguntas no ar:

1. **Se rodar de novo, o placar muda?** Muda. O modelo não responde igual duas vezes, e uma rodada só é uma foto tirada com a mão tremendo.
2. **Como saber se uma mudança piorou o agente?** Um colega encurta o prompt, outro troca o modelo por um mais barato. Em software comum, a suíte de testes responde em minutos. Aqui, a suíte precisa ser outra coisa.

Este notebook constrói essa outra coisa, camada por camada, sobre o agente da Aula 4.

### O que você vai sair sabendo

- Por que um caso de teste de agente tem uma **taxa** de acerto, e não um "passou/falhou".
- A **pirâmide de avaliação**: ferramenta, passo, trajetória, resultado e produção, e o que cada camada pega que as outras não pegam.
- Usar um **LLM como juiz** sem ser enganado por ele.
- Separar **regressão de ruído**: quando uma diferença no placar é real.
- Ler um atendimento como **trace com spans**: onde foi o tempo e onde foi o dinheiro.
- Definir **SLIs e SLOs** para um agente e usá-los como gate de deploy, com o custo por tarefa resolvida no centro.

### Mapa do notebook

| Parte | Pergunta | Camada da pirâmide |
|---|---|---|
| 0 · Ambiente | — | — |
| 1 · Não existe assert | Se eu rodar de novo, sai igual? | — |
| 2 · A pirâmide | O que exatamente está certo ou errado? | 1 a 4 |
| 3 · Teste A/A | Quanto o placar mexe sozinho? | — |
| 4 · O PR #42 | A mudança piorou o agente, ou é ruído? | regressão |
| 5 · Observabilidade | Onde foi o tempo, onde foi o dinheiro? E em produção, sem gabarito? | 5 |
| 6 · SLI, SLO e custo | Essa versão pode ir para produção? | gate |
| 7 · Síntese | — | — |

### O sistema avaliado

É o **agente único da Aula 4**, o agente que decide buscar: a busca do Encontro 3, atrás do contrato e da camada do Encontro 2, com as regras de resposta do Encontro 1. Ele aparece em três versões:

| Versão | O que mudou | Por que alguém faria isso |
|---|---|---|
| **v1 · produção** | nada: é o agente da Aula 4 | é a linha de base |
| **v2 · prompt enxuto (PR #42)** | um colega encurtou o prompt de sistema | economizar tokens e acabar com os loops |
| **v3 · modelo mais barato** | o mesmo prompt da v1, num modelo que custa menos por token | reduzir a conta |

### O golden set

São **25 casos com gabarito**: os 19 que vocês conhecem dos Encontros 1 a 4 e mais 6 que saíram dos traces da primeira semana de uso (`G1` a `G6`). O gabarito diz o desfecho certo, as páginas que a resposta precisa citar e, quando importa, o valor que precisa aparecer na resposta.

### Por que tudo roda sem chave

As respostas do modelo foram gravadas pelo professor, como no Encontro 4. A novidade é que **cada caso foi gravado cinco vezes**, nas três versões. Cada uma dessas repetições é uma **amostra**: a mesma pergunta, com o mesmo prompt, as mesmas ferramentas e o mesmo modelo, enviada de novo. É essa variação gravada que este notebook estuda. Tokens, custo e tempo de espera são os que cada chamada teve de verdade.

## Parte 0 · Ambiente

A célula abaixo baixa o repositório (no Colab), confere as gravações e roda as três versões nos 25 casos, cinco vezes cada: 375 atendimentos, reproduzidos da gravação em poucos segundos. Ao vivo, com o modelo, levaria mais de meia hora.

In [1]:
# Etapa 0 · No Colab, baixa o repositório da disciplina. Na sua máquina, não faz nada.
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.exists("/content/ai-agents-engineering"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/se-ai-agents-engineering/ai-agents-engineering-v2",
                        "/content/ai-agents-engineering"], check=True)
    os.chdir("/content/ai-agents-engineering/encontro-05")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-dotenv"], check=True)
sys.path.insert(0, os.getcwd())

import avaliacao as av
av.preparar("usar")
golden = av.carregar_golden()
v1 = av.rodar("v1", golden)
v2 = av.rodar("v2", golden)
v3 = av.rodar("v3", golden)
print(f"{len(golden)} casos x {av.AMOSTRAS} amostras x 3 versões = {len(v1['linhas']) * 3} atendimentos. Tudo certo, pode seguir.")

Gravação principal openai/gpt-4o-mini · US$ 0.15 / 0.6 por milhão de tokens (entrada / saída) · 2026-10-26 21:07
Gravação barato    google/gemini-2.5-flash-lite · US$ 0.1 / 0.4 por milhão de tokens (entrada / saída) · 2026-10-26 21:10
Gravação juiz      openai/gpt-4.1-mini · US$ 0.4 / 1.6 por milhão de tokens (entrada / saída) · 2026-10-26 21:11
Busca: 90 trechos da wiki da Aurora


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FileNotFoundError: Gravação não encontrada: C:\Users\cstefano\Desktop\Carlos\FIAP\Cursos\Engenharia de Software\AI Agents Engineering\Repositório agora vai\repo\encontro-05\dados\replay-barato.jsonl. O professor precisa rodar o gravar_replay.py.

## Parte 1 · Não existe assert

**Pergunta:** se eu fizer a mesma pergunta ao mesmo agente cinco vezes, recebo a mesma resposta?

Num teste de software comum, `assert soma(2, 2) == 4` passa ou falha, e passa ou falha **sempre**. A célula abaixo escolhe, na v1, o caso que mais variou entre as cinco amostras e mostra as cinco respostas.

In [ ]:
inst = av.acerto(v1)["instaveis"]
CASO = inst[0] if inst else "C1"
av.variacoes(v1, CASO)

**O que olhar.** Mesma pergunta, mesmo prompt, mesmas ferramentas, mesmo modelo. Mudam o texto, às vezes as fontes, às vezes o número de chamadas e, nos casos piores, o desfecho.

**Por que acontece.** O modelo escolhe cada palavra sorteando entre as mais prováveis; a temperatura controla o tamanho desse sorteio. Uma palavra diferente no começo leva a uma busca diferente, que traz trechos diferentes, que levam a outra resposta. Além disso, o mesmo modelo é servido por máquinas e provedores diferentes, e nem temperatura zero garante respostas idênticas.

**A consequência.** "Este caso passa" deixa de ser uma afirmação verdadeira ou falsa. Vira uma **taxa**: este caso passa 4 vezes em 5. Veja todos os casos instáveis da v1, um círculo por amostra (● acertou, ○ errou):

In [ ]:
av.instaveis(v1)

### Duas métricas, duas perguntas

| Métrica | Pergunta que responde | Como se calcula |
|---|---|---|
| **acerto** (pass@1) | Qual a chance de um atendimento sair certo? | a média, entre os casos, da fração de amostras certas |
| **pass^k** | Em quantos casos o agente acerta **sempre**? | a fração de casos que acertaram nas k amostras |

O acerto descreve o agente em média. O **pass^k** descreve a confiabilidade: um colaborador que pergunta a mesma coisa duas vezes e recebe duas respostas diferentes perde a confiança no atendimento, mesmo que uma delas esteja certa. Quanto maior o k, mais o pass^k cai, e é ele que importa num agente que atende muita gente.

O placar deste encontro traz as duas, mais o **intervalo de confiança** do acerto: a faixa onde o acerto verdadeiro provavelmente está, considerando que 25 casos são uma amostra pequena (Parte 3).

In [ ]:
av.placar(v1)

## Parte 2 · A pirâmide de avaliação

**Pergunta:** quando o atendimento sai errado, **o que** está errado?

Um placar de "acertou ou errou" diz que algo quebrou, mas não onde. A pirâmide separa a avaliação em camadas, da mais barata e rápida (embaixo) à mais cara e mais próxima do usuário (em cima):

| Camada | Pergunta | Precisa do modelo? | No caso da Aurora |
|---|---|---|---|
| **1 · Unit de ferramenta** | A ferramenta faz o que promete? | não | a busca devolve a página certa? |
| **2 · Step-level** | O agente chamou a ferramenta certa, com os argumentos certos? | sim | levou o "809" para a busca? |
| **3 · Trajetória** | O caminho até a resposta foi razoável? | sim | buscou 6 vezes a mesma coisa? |
| **4 · Outcome** | O resultado resolve a tarefa? | sim | desfecho, fontes e valor certos? |
| **5 · Online** | O que acontece com tráfego real? | sim, e sem gabarito | quantos vão para o RH? quanto custa? |

A regra prática é a da pirâmide de testes que vocês já conhecem: muita coisa embaixo, onde é barato e o erro aponta o culpado, e pouca coisa em cima, onde é caro e o erro só diz que algo quebrou.

### 2.1 · Unit de ferramenta: a busca sozinha

Sem modelo, sem custo, sem variação: a busca recebe a pergunta do colaborador e devolve três trechos. A página que o gabarito exige está entre eles? É o placar da nota de chunking do Encontro 3, agora como o degrau de baixo da pirâmide. Esta camada roda em segundos e pode rodar a cada commit.

In [ ]:
busca_unit = av.avaliar_busca(golden)

**Repare:** quando a busca falha aqui, nenhum prompt conserta o atendimento lá em cima. Quando ela passa e o atendimento falha, o problema está nas camadas de cima. Um erro nesta camada aponta o culpado; um erro na camada 4 só diz que algo quebrou.

### 2.2 · Step-level: cada passo do agente

Aqui olhamos as decisões do agente, uma por uma, sem olhar para a resposta final:

- **buscou quando devia:** busca nas dúvidas de política e não busca em "bom dia" e "obrigado";
- **termo exato:** nos casos T1 e T2, levou o código (`809`, `AUR-S2`) para a primeira busca? Se trocar por "erro de conexão", a parte léxica da busca híbrida perde o que tinha de melhor;
- **espaço certo:** quando usou o filtro opcional de espaço, filtrou o espaço onde está a resposta?

In [ ]:
_ = av.passos(v1, golden)

### 2.3 · Trajetória: o caminho

A resposta pode sair certa por um caminho caro: seis buscas, duas iguais, até achar. E pode sair errada porque o caminho não terminou: o agente buscou até bater no **limite de passos** (o critério de parada do Encontro 1) e foi encaminhado ao RH sem resposta.

In [ ]:
_ = av.trajetoria(v1)

**O que olhar.** Os atendimentos que bateram no limite são o sintoma de um loop: o agente não encontra a informação e não sabe parar. Em geral, são as perguntas **sem resposta na wiki**: o "não sei" é um comportamento que precisa ser projetado, e o limite de passos é a rede de proteção quando o projeto falha. A Parte 5 abre um desses atendimentos por dentro.

### 2.4 · Outcome com régua

É a régua dos Encontros 1 a 4: desfecho, fontes obrigatórias e, quando o caso pede, o valor certo na resposta (`20 dias`, `R$ 160`) e a ausência do valor errado (`R$ 80`, da política antiga). Ela é **determinística, barata e rápida**, e é por isso que é a régua principal deste encontro.

Mas ela tem pontos cegos. Ela não vê:

- uma resposta que traz o valor certo e **inventa** uma condição que não está na política;
- uma resposta certa escrita de um jeito que o gabarito não previu ("vinte dias corridos" quando o gabarito só aceita "20 dias");
- a qualidade de uma resposta para uma pergunta que **não tem gabarito**, que é o caso de tudo o que chega em produção.

Para esses, entra o juiz.

### 2.5 · Outcome com LLM-as-judge

Um **juiz** é outro modelo, que lê o atendimento e dá um veredito. É a forma de avaliar texto livre em escala. Aqui há dois juízes, de propósito, gravados com um modelo diferente e mais forte que o avaliado:

In [ ]:
print("JUIZ INGÊNUO\n" + av.JUIZ_INGENUO)
print("\n" + "=" * 100 + "\n\nJUIZ COM EVIDÊNCIA\n" + av.JUIZ_EVIDENCIA)

O **ingênuo** é o juiz que quase todo time escreve na primeira vez: lê a pergunta e a resposta e dá uma nota. O **com evidência** recebe também os trechos que o agente leu e segue uma rubrica com três critérios, um de cada vez, com a justificativa antes do veredito.

Primeiro, o ingênuo contra a régua, nas 125 respostas da v1:

In [ ]:
av.julgar(v1, "ingenuo")
discord_ingenuo = av.concordancia(v1, "ingenuo")

**Pergunta para a turma (enquete).** O juiz aprovou respostas que a régua reprova. Antes de olhar: quem vocês acham que está certo nessas discordâncias, o juiz ou a régua?

In [ ]:
av.mostrar_discordancias([l for l in discord_ingenuo if not l["acertou"]], "ingenuo", maximo=3)

**O que o juiz ingênuo erra, e por quê.** Ele não sabe qual é a política da Aurora, então julga se a resposta **parece** boa: clara, educada, confiante. Um valor inventado, dito com segurança, passa. Os vieses conhecidos de juízes de LLM vão nessa direção: preferem respostas longas, confiantes e parecidas com o que eles mesmos escreveriam.

Agora o juiz com evidência, nas mesmas respostas:

In [ ]:
av.julgar(v1, "evidencia")
discord_evidencia = av.concordancia(v1, "evidencia")
print()
av.mostrar_discordancias(discord_evidencia, "evidencia", maximo=3)

**Como ler as discordâncias que sobraram.** Com evidência e rubrica, o juiz concorda muito mais com a régua. Nas discordâncias que restam, às vezes o juiz erra, e às vezes **a régua erra**: uma resposta certa que o gabarito não previu, ou uma resposta com o valor certo e uma condição inventada que a régua deixou passar.

### Como confiar num juiz

1. **Dê evidência e rubrica.** Um juiz sem os trechos julga estilo, não verdade.
2. **Peça a justificativa antes do veredito,** e critérios separados em vez de uma nota de 1 a 5.
3. **Calibre contra humanos antes de confiar no número.** Separe algumas dezenas de respostas, peça a quem conhece o domínio para marcar certo ou errado, e meça a concordância do juiz com essas marcações. Se ela for baixa, o painel do juiz é ruído com boa aparência.
4. **Use um modelo diferente do avaliado.** Um modelo tende a aprovar o próprio estilo.
5. **Lembre que o juiz também custa.** A célula abaixo mostra quanto custou julgar a rodada.

**Pergunta para a turma (enquete).** Nas três discordâncias acima, quem está certo? Vocês acabaram de fazer o passo 3: a turma é o humano que calibra o juiz.

In [ ]:
custo_agente = sum(l["custo_usd"] for l in v1["linhas"])
custo_juiz = sum(l["juiz_evidencia"]["custo_usd"] for l in v1["linhas"])
print(f"atender os 125 casos da v1: US$ {custo_agente:.4f}")
print(f"julgá-los com evidência:    US$ {custo_juiz:.4f}  ({custo_juiz / custo_agente:.0%} do custo de atender)")

## Parte 3 · Quanto o placar mexe sozinho: o teste A/A

**Pergunta:** se eu rodar a suíte inteira de novo, **sem mudar nada**, o acerto muda quanto?

Em experimentos com usuários, um **teste A/A** compara uma versão com ela mesma, para medir o tamanho do acaso antes de comparar duas versões diferentes. Aqui, cada uma das cinco amostras é uma rodada completa da suíte com a v1, sem mudar uma vírgula.

In [ ]:
ruido = av.teste_aa(v1)

**A lição.** Se a mesma versão, rodada de novo, já muda o placar nesse tanto, então **uma diferença desse tamanho entre duas versões não prova nada**. É o nível de ruído da suíte. Qualquer decisão do tipo "a nova versão é melhor porque acertou 2 casos a mais" precisa passar por esse filtro.

### O intervalo de confiança

O placar mostra o acerto com um **intervalo de 90%**. Ele foi calculado por *bootstrap*: o computador sorteia milhares de golden sets "parecidos" com o nosso, reamostrando os 25 casos com reposição e, dentro de cada caso, as 5 amostras com reposição. Calcula o acerto de cada sorteio e guarda a faixa onde ficaram 90% deles. Assim, o intervalo carrega as duas incertezas: a de ter poucos casos e a de o modelo variar. Em palavras: com esta suíte, o acerto verdadeiro do agente provavelmente está nessa faixa.

A comparação entre duas versões (Parte 4) usa a mesma ideia, **pareada**: cada caso é comparado com ele mesmo na outra versão, o que tira da conta a diferença de dificuldade entre os casos.

Duas formas de estreitar o intervalo: **mais casos** no golden set (é o que o hands-on faz) e **mais amostras** por caso. As duas custam, e é por isso que o tamanho do golden set é uma decisão de engenharia, não um detalhe.

## Parte 4 · O PR #42: regressão ou ruído?

**Pergunta:** um colega abriu um pull request que encurta o prompt. A suíte diz que piorou, melhorou, ou que não dá para saber?

A descrição do PR diz: *"Prompt 60% menor, respostas mais diretas, e o 'busque uma vez' acaba com os loops que vimos nos traces."* O diff:

In [ ]:
print("ANTES (v1)\n" + av.VERSOES["v1"]["sistema"])
print("\n" + "=" * 100 + "\n\nDEPOIS (v2, PR #42)\n" + av.VERSOES["v2"]["sistema"])

**Pergunta para a turma (enquete), antes de rodar qualquer coisa.** Vocês aprovariam esse PR olhando só o diff? O que sumiu do prompt que poderia fazer falta?

### O que um time faria sem esta aula

Rodaria a suíte **uma vez** em cada versão e compararia os dois números:

In [ ]:
av.placar(av.recorte(v1, 0), av.recorte(v2, 0))

Com uma rodada de cada, a decisão sai por impulso: "subiu, aprova" ou "caiu, rejeita". Agora, a comparação feita direito: as cinco amostras, caso a caso, pareando cada caso consigo mesmo, com o intervalo da diferença.

In [ ]:
cmp_pr = av.comparar(v1, v2)

**Como ler o veredito.**

- **Regressão:** o intervalo da diferença está inteiro abaixo de zero. A versão nova é pior, e não é acaso.
- **Melhora:** o intervalo está inteiro acima de zero.
- **Não dá para afirmar:** o intervalo cruza o zero. A diferença média pode ser só ruído, do tamanho do teste A/A.

Mas o veredito médio é só metade da história. **A média esconde.** Um PR pode melhorar oito casos fáceis e quebrar um caso crítico, e a média sobe. O mapa abaixo mostra, caso a caso, quantas das cinco amostras acertaram em cada versão:

In [ ]:
av.mapa_de_casos(v1, v2)

**O que olhar.** As linhas em que a cor muda muito entre as colunas. Procurem as categorias: o PR tirou do prompt a regra de elegibilidade (`X1`, `G6`) e a regra de conflito, e trocou "pode buscar de novo" por "busque uma vez" (`R1`, perguntas de duas partes). As perguntas sem resposta (`S1`, `S2`, `G5`) talvez tenham melhorado: era o que o PR prometia.

A célula abaixo abre o caso que mais piorou, se houver um:

In [ ]:
CASO_PIOR = cmp_pr["piores"][0] if cmp_pr["piores"] else (cmp_pr["melhores"][0] if cmp_pr["melhores"] else "X1")
av.variacoes(v2, CASO_PIOR)

### O gate de regressão

Num time, essa comparação roda sozinha a cada PR que mexe em prompt, modelo ou ferramenta, e bloqueia o merge quando falha. É o **versionamento de prompt tratado como versionamento de código**, com revisão e teste. Um gate razoável tem duas regras:

1. **A média não pode cair além do ruído:** o intervalo da diferença não pode estar inteiro abaixo de zero.
2. **Os casos críticos não podem cair, nem que a média suba:** aqui, o escalonamento de elegibilidade (`X1`, `G6`) e as perguntas sem resposta, porque uma resposta inventada sobre benefício é exatamente o terceiro problema de arquitetura do caso da Aurora, a auditabilidade.

O notebook 03 mostra esse gate numa plataforma de mercado.

**Pergunta para a turma.** Com o que vocês viram, o PR #42 entra? Se não entra, o que vocês pediriam ao colega para mudar, mantendo a economia?

## Parte 5 · Observabilidade: o atendimento por dentro

**Pergunta:** quando um atendimento demora ou custa caro, onde foram o tempo e o dinheiro?

### Do log ao trace, do trace ao span

| Conceito | O que é | No agente |
|---|---|---|
| **log** | uma linha solta com um evento | "chamou a busca" |
| **trace** | tudo o que aconteceu num pedido, ligado por um id | o atendimento inteiro, da pergunta à resposta |
| **span** | um intervalo de tempo dentro do trace, com início, fim e atributos | uma chamada ao modelo, uma busca |

No Encontro 4, o `mostrar_trace` listava eventos. O que as plataformas de observabilidade (Langfuse, LangSmith, Arize Phoenix) e o OpenTelemetry fazem é transformar esses eventos em **spans aninhados**: um span raiz para o atendimento e, dentro dele, um span por passo, cada um com duração, tokens e custo. A célula abaixo faz isso com o atendimento **mais demorado** da v1:

In [ ]:
LENTO, AMOSTRA_LENTA = av.mais_lento(v1)
_ = av.cascata(v1, LENTO, AMOSTRA_LENTA)

**Como ler a cascata (*waterfall*).**

- Cada barra azul é uma chamada ao modelo; cada traço verde é uma busca. A busca leva milissegundos; o modelo, segundos. **Otimizar a busca não muda a latência do agente. Cortar passos muda.**
- O custo de cada chamada cresce a cada passo, porque o histórico inteiro vai de novo para o modelo: o passo 5 lê tudo o que os passos 1 a 4 leram, mais as respostas da busca. Um loop não custa linear, custa cada vez mais.
- O trace id é o que permite responder à pergunta de auditoria do caso: "por que o agente disse isso?". Sem ele, a resposta não existe.

Para comparar com o que vocês viram no Encontro 4:

In [ ]:
av.mostrar_trace(v1, LENTO, AMOSTRA_LENTA)

### O painel

O painel junta as quatro perguntas de operação, uma cor por versão: está certo? quanto custa uma resposta certa? quanto o colaborador espera? quantos atendimentos entram em loop?

In [ ]:
av.painel(v1, v2, v3)

**Duas camadas de observabilidade, complementares.** A plataforma de observabilidade de LLM (Langfuse, LangSmith) cuida dos traces do agente, dos evals e das métricas de modelo: tokens, custo, qualidade. O APM de infraestrutura (Datadog, Grafana, New Relic) cuida do host, dos erros da aplicação e da saúde do deploy. Um agente em produção precisa das duas; uma não substitui a outra.

### Camada 5 · Online: produção, sem gabarito

Em produção, ninguém escreve o gabarito antes de o colaborador perguntar. As 24 perguntas abaixo são da primeira semana de uso: a v1 atendeu cada uma cinco vezes. O que dá para medir?

In [ ]:
banco = av.carregar_banco()
producao = av.rodar("v1", banco)
av.julgar(producao, "evidencia")
av.online(producao)

**O que dá e o que não dá.** Desfechos, loops, latência e custo saem de graça, do próprio trace. **Qualidade, não.** Sem gabarito, o único sinal de qualidade é um juiz (que precisa estar calibrado, Parte 2) ou o colaborador (o 👍/👎, a pergunta repetida, o chamado aberto depois).

E é daqui que o golden set cresce: um trace de produção estranho vira um caso novo, com o gabarito escrito por quem conhece a política. Os casos `G1` a `G6` nasceram assim. **É exatamente o que vocês vão fazer no hands-on.**

## Parte 6 · SLI, SLO e custo

**Pergunta:** essa versão pode ir para produção?

### SLI, SLO e orçamento de erro

| Conceito | O que é | Exemplo na Aurora |
|---|---|---|
| **SLI** (indicador) | uma medida do serviço, como uma fração ou um percentil | fração dos atendimentos que bateram no limite de passos |
| **SLO** (objetivo) | o alvo para o SLI, combinado com o negócio | no máximo 2% |
| **orçamento de erro** | o quanto se pode falhar antes de violar o SLO | 2 em cada 100 atendimentos por semana |

O orçamento de erro é o que transforma o SLO em decisão: enquanto sobra orçamento, o time pode arriscar mudanças; quando acaba, congela as mudanças e cuida da confiabilidade. Os SLIs que este módulo mede:

In [ ]:
av.listar_slis()

### FinOps: o preço do token engana

A v3 usa um modelo mais barato por token. Ela é mais barata **por resposta certa**? O número que importa é o **custo por tarefa resolvida**: um modelo que custa metade por token mas erra mais, ou que precisa de mais passos para chegar à resposta, pode sair mais caro por acerto. É o mesmo raciocínio do multi-agente do Encontro 4, que custava muito mais pelo mesmo desfecho.

In [ ]:
av.placar(v1, v3)
print()
_ = av.comparar(v1, v3)

### O gate de deploy

Um SLO só vale se alguém o usa para decidir. A célula abaixo é um gate: um alvo para cada SLI escolhido, e a pergunta "pode ir para produção?" para cada versão. Os valores são um ponto de partida. **Mudem e rodem de novo.**

In [ ]:
SLOS = {
    "acerto": 0.80,                 # pelo menos 80% dos atendimentos do golden set certos
    "bateu_limite": 0.02,           # no máximo 2% dos atendimentos em loop
    "p95_segundos": 10.0,           # 95% dos colaboradores esperam no máximo 10 segundos
    "custo_por_acerto": 0.002,      # no máximo US$ 0,002 por resposta certa (US$ 2 a cada mil)
}
_ = av.checar_slos(SLOS, v1, v2, v3)

**Perguntas para a turma.**

1. Qual versão vai para produção? E se nenhuma passar no gate, o que se faz: muda o agente ou muda o SLO?
2. Quem define o número do SLO de acerto: o time de engenharia, o RH ou o jurídico? O que muda se o agente responder sobre benefício, e não sobre senha de Wi-Fi?
3. O acerto vem de 25 casos e tem um intervalo largo. Um SLO de 80% faz sentido se o intervalo vai de 70% a 90%?

## Parte 7 · Síntese

### O que cada camada mostrou hoje

| Camada | O que pegou | Custo para rodar |
|---|---|---|
| 1 · Unit de ferramenta | se a busca traz a página certa, sem modelo nenhum | zero |
| 2 · Step-level | termo exato perdido, busca onde não precisava | o do agente |
| 3 · Trajetória | loops até o limite de passos, buscas repetidas | o do agente |
| 4 · Outcome (régua) | desfecho, fonte e valor errados | o do agente |
| 4 · Outcome (juiz) | resposta inventada, o que a régua não vê; com o risco de ser enganado | o do agente + o do juiz |
| 5 · Online | desfechos, loops, latência e custo sem gabarito; qualidade só com juiz ou com o usuário | o de produção |

### O mapa da disciplina, na linha deste encontro

| Prática consolidada | Equivalente no mundo de agentes |
|---|---|
| Teste unitário com assert | **Eval sobre golden set, com taxa de acerto e intervalo** |
| Log de aplicação | **Trace com span por passo, tool call e token** |
| Custo de infraestrutura | **Custo por tarefa resolvida, não por token** |
| Code review | **Revisão de prompt com gate de regressão** |

### Três frases para levar

1. **Uma rodada não é uma medida.** Antes de comparar duas versões, meça quanto a mesma versão varia sozinha.
2. **A média esconde.** O gate de deploy olha os casos críticos, não só o acerto médio.
3. **Um juiz sem evidência julga estilo.** Dê os trechos, dê a rubrica, e calibre contra humanos antes de confiar.

### Ponte para o Encontro 6

O agente agora está medido: sabemos quanto ele acerta, quanto varia, quanto custa e onde falha. Falta a pergunta que nenhum golden set responde sozinho: **o que acontece quando alguém tenta fazê-lo errar de propósito?** Uma página da wiki editável por qualquer pessoa é uma porta de entrada. É o Encontro 6.